# 🚀 Aurora Siger — Sistema de Análise de Telemetria

Sistema de verificação pré-lançamento com análise por IA (Google Gemini).

---
**Instruções:**
1. Execute a célula de instalação (Célula 1)
2. Insira sua chave da API Gemini (Célula 2)
3. Preencha os dados de telemetria (Célula 3)
4. Execute a análise (Célula 4)

In [ ]:
# Célula 1 — Instalação de dependências
!pip install -q google-generativeai ipywidgets

In [ ]:
# Célula 2 — Configuração da API Key
import os
import getpass
import google.generativeai as genai

# Tenta ler do ambiente (ex: Colab Secrets), caso contrário pede digitação
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ API Key carregada via Colab Secrets.")
except Exception:
    GEMINI_API_KEY = getpass.getpass("🔑 Insira sua Gemini API Key: ")

genai.configure(api_key=GEMINI_API_KEY)
modelo = genai.GenerativeModel("gemini-2.5-flash")
print("✅ Modelo Gemini configurado.")

In [ ]:
# Célula 3 — Entrada de dados de telemetria via widgets
import ipywidgets as widgets
from IPython.display import display

style  = {'description_width': '260px'}
layout = widgets.Layout(width='480px')

w_temp_interna      = widgets.FloatSlider(value=22.0,  min=-10,  max=40,  step=0.5,  description='Temperatura interna (°C):',      style=style, layout=layout)
w_temp_externa      = widgets.FloatSlider(value=20.0,  min=-5,   max=45,  step=0.5,  description='Temperatura externa (°C):',      style=style, layout=layout)
w_nivel_energia     = widgets.FloatSlider(value=85.0,  min=0,    max=100, step=1.0,  description='Nível de energia (%):',          style=style, layout=layout)
w_nivel_combustivel = widgets.FloatSlider(value=90.0,  min=0,    max=100, step=1.0,  description='Nível de combustível RP-1 (%):',style=style, layout=layout)
w_nivel_oxidante    = widgets.FloatSlider(value=90.0,  min=0,    max=100, step=1.0,  description='Nível de oxidante LOX (%):',    style=style, layout=layout)
w_pressao_tanque    = widgets.FloatSlider(value=3.5,   min=0,    max=6,   step=0.1,  description='Pressão do tanque (bar):',      style=style, layout=layout)
w_integridade       = widgets.ToggleButtons(value='OK', options=['OK', 'ERRO'],       description='Integridade estrutural:',       style=style)

print("=" * 55)
print("     AURORA SIGER — Dados de Telemetria")
print("=" * 55)
display(
    w_temp_interna, w_temp_externa,
    w_nivel_energia, w_nivel_combustivel, w_nivel_oxidante,
    w_pressao_tanque, w_integridade
)

In [ ]:
# Célula 4 — Verificações e análise IA
import time
from IPython.display import Markdown

# Leitura dos widgets
temp_interna      = w_temp_interna.value
temp_externa      = w_temp_externa.value
nivel_energia     = w_nivel_energia.value
nivel_combustivel = w_nivel_combustivel.value
nivel_oxidante    = w_nivel_oxidante.value
pressao_tanque    = w_pressao_tanque.value
integridade       = 1 if w_integridade.value == 'OK' else 0

gradiente_termico = abs(temp_interna - temp_externa)

# Verificações
resultados = []

checks = [
    ("Temperatura interna",       -10 <= temp_interna <= 40,       f"Fora do limite ({temp_interna}°C). Esperado: -10°C a 40°C."),
    ("Temperatura externa",       -5  <= temp_externa <= 45,       f"Fora do limite ({temp_externa}°C). Esperado: -5°C a 45°C."),
    ("Gradiente térmico",         gradiente_termico <= 40,          f"Gradiente excessivo ({gradiente_termico:.1f}°C). Máx: 40°C."),
    ("Nível de energia",          nivel_energia >= 70,              f"Energia insuficiente ({nivel_energia:.1f}%). Mín: 70%."),
    ("Nível de combustível RP-1", nivel_combustivel >= 80,          f"Combustível insuficiente ({nivel_combustivel:.1f}%). Mín: 80%."),
    ("Nível de oxidante LOX",     nivel_oxidante >= 80,             f"Oxidante insuficiente ({nivel_oxidante:.1f}%). Mín: 80%."),
    ("Pressão do tanque",         2.5 <= pressao_tanque <= 4.5,    f"Pressão fora do intervalo ({pressao_tanque:.2f} bar). Esperado: 2.5–4.5 bar."),
    ("Integridade estrutural",    integridade == 1,                 "Falha estrutural detectada."),
]

print("\n" + "=" * 55)
print("     AURORA SIGER — Verificação de Sistemas")
print("=" * 55)

todas_ok = True
for nome, passou, erro in checks:
    if passou:
        print(f"  >> {nome}: OK")
    else:
        print(f"  >> {nome}: FALHA")
        print(f"     ⚠ {erro}")
        todas_ok = False

print("\n" + "=" * 55)
if todas_ok:
    print("\n✅ PRONTO PARA DECOLAR")
else:
    print("\n❌ DECOLAGEM ABORTADA")
print("=" * 55)

# Análise Gemini
status_texto = "PRONTO PARA DECOLAR" if todas_ok else "DECOLAGEM ABORTADA"

prompt = f"""Você é um sistema de análise de telemetria aeroespacial. Analise os dados da Missão Aurora Siger:

TELEMETRIA:
- Temperatura interna: {temp_interna}°C
- Temperatura externa: {temp_externa}°C
- Gradiente térmico: {gradiente_termico:.1f}°C
- Nível de energia: {nivel_energia:.1f}%
- Nível de combustível (RP-1): {nivel_combustivel:.1f}%
- Nível de oxidante (LOX): {nivel_oxidante:.1f}%
- Pressão do tanque: {pressao_tanque:.2f} bar
- Integridade estrutural: {"OK" if integridade == 1 else "ERRO"}
- Status geral: {status_texto}

Responda em português com:
1. Classificação de cada dado (normal / atenção / crítico)
2. Possíveis anomalias identificadas
3. Sugestões de risco
4. Formato de resposta: Texto simples, direto e claro. Sem formatação adicional.
"""

print("\n[ Análise IA — Gemini ] Aguardando resultado...\n")
resposta = modelo.generate_content(prompt)
print(resposta.text)